<a href="https://colab.research.google.com/github/robertbarcik/ADK-tutorial/blob/main/notebooks/07_callbacks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 07 — Callbacks: Your Code Before and After Every Step

> **Where you are** — your agents work; now you wrap controls around them.
> - **You can already:** pass a function around as a value — your `available_functions = {"get_current_weather": get_current_weather}` dict from the previous course stored functions without calling them. Callbacks are the same move: you hand ADK a function (no `()` after the name!), and ADK calls it at the right moment.
> - **New in this module:** the six hook points and the return-to-override rule.

Picture the help desk agent from earlier modules running in a real company. Sooner or later someone asks it for a password, a tool returns an employee's salary, or you want to test the agent without hitting a paid API. In all three cases you need to step in **between** the agent's steps — without rewriting the agent.

That is exactly what a **callback** is: a plain Python function you hand to the agent, and ADK runs it at a fixed moment — just before the model is called, just after a tool returns, and so on. Think of it as a security check at a door: everything passing through that door goes past your code first.

**What we'll build:** a blocklist that stops bad requests before the model sees them, a redactor that strips private fields from tool results, and a mock that replaces a real API during tests.

**Running cost:** under $0.01.

## The Six Doors

There are three moments where ADK can hand control to you — the agent starting and finishing, each model call, each tool call — and each moment has a *before* and an *after* door:

| Event | Before | After |
|---|---|---|
| Agent runs | `before_agent_callback` | `after_agent_callback` |
| Model call | `before_model_callback` | `after_model_callback` |
| Tool call | `before_tool_callback` | `after_tool_callback` |

(Plus two doors for error recovery: `on_model_error_callback` and `on_tool_error_callback`.) We'll use three of them today; the rest work exactly the same way.

## One Rule Runs Everything: Return to Override

Every callback follows the same contract:

```python
def my_callback(context, request_or_response_or_args):
    # Observe, log, check — whatever you need.
    if <some condition>:
        return <a replacement value>   # ← ADK uses this; the real call is skipped
    return None                        # ← the real call proceeds
```

`None` means "carry on." A returned object means "use this instead of doing the thing."

That is the entire callback API. Guardrails, caches, mocks, redactors — all the same rule, different return values.

# Setup

Same ritual as every module: install, key, imports.

In [1]:
!pip install -q google-adk==2.7.1 litellm==1.85.7 python-dotenv==1.0.1 nest-asyncio==1.6.0 deprecated==1.2.18 2>/dev/null
print("✅ Packages installed.")

✅ Packages installed.


## API Key

In [2]:
import os
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
    print("✅ API key loaded from Colab secrets.")
except Exception:
    try:
        from dotenv import load_dotenv; load_dotenv()
        OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
        if OPENROUTER_API_KEY: print("✅ API key loaded from .env file.")
    except ImportError: pass
if not OPENROUTER_API_KEY:
    from getpass import getpass
    OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")
assert OPENROUTER_API_KEY, "❌ No API key provided."
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
MODEL_STRING = "openrouter/openai/gpt-5.6-luna"
print(f"✅ Model: {MODEL_STRING}")

✅ API key loaded from .env file.
✅ Model: openrouter/openai/gpt-5.6-luna


## Imports

One new import next to the usual ones: `LlmResponse` — the object a callback returns when it wants to answer *instead of* the model.

In [3]:
import os
import sys, warnings, asyncio, uuid, logging
warnings.filterwarnings("ignore")
try: sys.stderr.fileno()
except Exception: sys.stderr = open(os.devnull, "w")
import nest_asyncio; nest_asyncio.apply()
os.environ.setdefault("LITELLM_LOG", "ERROR")  # silence LiteLLM's import-time provider warnings
import litellm; litellm.suppress_debug_info = True
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.models.lite_llm import LiteLlm
from google.adk.models.llm_response import LlmResponse
from google.genai import types

print("✅ Imports successful.")

✅ Imports successful.


# The `chat()` Helper Again

Same helper as every module (session → runner → events). Nothing new — run it and move on.

In [4]:
APP = "m07_callbacks"
USER = "student"
session_service = InMemorySessionService()

async def chat(agent, prompt: str, label: str = ""):
    sid = f"s-{uuid.uuid4().hex[:6]}"
    await session_service.create_session(app_name=APP, user_id=USER, session_id=sid)
    runner = Runner(agent=agent, app_name=APP, session_service=session_service)
    msg = types.Content(role="user", parts=[types.Part(text=prompt)])
    prefix = f"[{label}] " if label else ""
    print(f"{prefix}USER: {prompt}")
    async for ev in runner.run_async(user_id=USER, session_id=sid, new_message=msg):
        if ev.content and ev.content.parts:
            for p in ev.content.parts:
                if p.text and p.text.strip():
                    tag = "[FINAL]" if ev.is_final_response() else "[step]"
                    print(f"{prefix}{tag} {ev.author}: {p.text.strip()[:200]}")
                if p.function_call:
                    args = dict(p.function_call.args) if p.function_call.args else {}
                    print(f"{prefix}[tool_call] {p.function_call.name}({args})")
                if p.function_response:
                    print(f"{prefix}[tool_resp] {p.function_response.response}")
    print()

print("✅ chat() ready.")

✅ chat() ready.


# A Guardrail Before the Model: The Blocklist

First thing to worry about: *"What's the CEO's password?"* You could write "refuse such questions" into the instruction — but an instruction is a polite request the model can ignore. A **`before_model_callback`** is enforcement: your function runs before *every* model call, sees the full request that is about to go out, and can stop it.

The plan: keep a list of forbidden words. If the latest user message contains one, answer with a canned refusal — and the model is never called (zero tokens billed). If not, return `None` and everything proceeds normally.

### The key lines, before you run them

```python
def blocklist_guardrail(callback_context, llm_request):
```

You never call this function yourself. ADK calls it and fills in both arguments — the same move as `tool_context` earlier in the course. `llm_request` carries everything about to be sent: history, instruction, tool definitions.

```python
before_model_callback=blocklist_guardrail,
```

Note: **no parentheses**. `blocklist_guardrail()` would run the function right now and pass its result; `blocklist_guardrail` passes *the function itself*, for ADK to call later — exactly how you stored functions in your `available_functions` dict in the previous course.

And the refusal we return is an `LlmResponse` wrapping `types.Content(role="model", parts=[...])` — the same Content/Part envelope our `chat()` helper builds for user messages, just written for the model's side of the conversation.

In [5]:
BLOCKED_WORDS = ["password", "secret", "credit card"]

def blocklist_guardrail(callback_context, llm_request):
    """Short-circuits the LLM if the latest user message contains a blocked word."""
    # llm_request.contents is the chat history, latest turn last.
    latest_user = ""
    if llm_request.contents:
        for part in llm_request.contents[-1].parts or []:
            if part.text:
                latest_user = part.text.lower()
                break

    for bad in BLOCKED_WORDS:
        if bad in latest_user:
            # Return a canned response. ADK uses this; the LLM is not called.
            return LlmResponse(
                content=types.Content(
                    role="model",
                    parts=[types.Part(text=(
                        f"I can't help with requests involving '{bad}'. "
                        f"Please rephrase your question."
                    ))],
                )
            )
    return None  # No block; proceed normally.

guarded_agent = LlmAgent(
    name="guarded_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="A helpful agent with a blocklist guardrail.",
    instruction="You are helpful and concise. Answer in one short paragraph.",
    before_model_callback=blocklist_guardrail,
)

await chat(guarded_agent, "What is photosynthesis?", "normal")
await chat(guarded_agent, "What's the CEO's password?", "blocked")

[normal] USER: What is photosynthesis?


[normal] [FINAL] guarded_agent: Photosynthesis is the process by which plants, algae, and some bacteria use sunlight to convert water and carbon dioxide into glucose, a form of stored chemical energy, and oxygen. It mainly occurs in

[blocked] USER: What's the CEO's password?
[blocked] [FINAL] guarded_agent: I can't help with requests involving 'password'. Please rephrase your question.



### 🔍 What just happened?

- First run: no blocked words → the callback returned `None` → the LLM ran → a normal answer.
- Second run: "password" matched → the callback returned its canned `LlmResponse` → **the LLM was never called**. No `[step]` events, zero tokens billed.

Swap the `in` check for a regex, a PII classifier, or a toxicity model — the shape stays the same. An instruction is a polite request; this check is a wall.

### 🎯 Mini-tasks

1. **A caching callback.** Cache answers keyed on the last user message: on a hit, return a canned `LlmResponse` with the stored answer; on a miss, return `None`. Verify that repeating a question doesn't call the LLM.
2. **A length limiter.** Write an `after_model_callback` that cuts any response over 200 characters and appends "...". Test it with "write me a long essay".

# Redact PII After a Tool

Next worry: the help desk looks up an employee, and the HR tool returns *everything* — salary, personal ID, home address. The model doesn't need those to answer "what's Alice's email?", and whatever enters the model's context can end up in its answer.

An **`after_tool_callback`** runs right after a tool returns, *before* the result reaches the model. Return `None` to pass the result through; return a replacement and the model sees that instead. (The tool has already run — you can't undo its side effects — but you decide what the model learns.)

One decode before the code:

```python
def redact_pii(tool, args, tool_context, tool_response):
```

Four arguments, all filled in by ADK: which tool ran, the arguments it got, the context, and — the one we care about here — its return value.

In [6]:
# A fake HR database — includes sensitive fields.
def lookup_employee(name: str) -> dict:
    """Look up an employee by name."""
    DB = {
        "Alice": {"name": "Alice", "email": "alice@company.com",
                  "department": "Engineering",
                  "salary": "72000 EUR", "ssn": "881234567",
                  "home_address": "Main St 42, Bratislava"},
        "Bob":   {"name": "Bob",   "email": "bob@company.com",
                  "department": "Marketing",
                  "salary": "65000 EUR", "ssn": "921112233",
                  "home_address": "Oak Ave 8, Kosice"},
    }
    return DB.get(name, {"error": f"No employee named {name}."})

SENSITIVE_FIELDS = {"salary", "ssn", "home_address", "date_of_birth"}

def redact_pii(tool, args, tool_context, tool_response):
    """Replace sensitive fields with [REDACTED] before the model sees them."""
    if isinstance(tool_response, dict):
        cleaned = dict(tool_response)
        for key in SENSITIVE_FIELDS & cleaned.keys():
            cleaned[key] = "[REDACTED]"
        return cleaned
    return None  # No change

hr_agent = LlmAgent(
    name="hr_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Answers employee-info questions with PII redaction.",
    instruction="Use lookup_employee to answer. Report what you can see.",
    tools=[lookup_employee],
    after_tool_callback=redact_pii,
)

await chat(hr_agent, "Look up Alice's contact info and tell me what you know about her.")

USER: Look up Alice's contact info and tell me what you know about her.


[tool_call] lookup_employee({'name': 'Alice'})
[tool_resp] {'name': 'Alice', 'email': 'alice@company.com', 'department': 'Engineering', 'salary': '[REDACTED]', 'ssn': '[REDACTED]', 'home_address': '[REDACTED]'}


[FINAL] hr_agent: Alice’s contact information:
- Email: alice@company.com
- Department: Engineering

I can’t share sensitive personal information such as salary, SSN, or home address.



### 🔍 What just happened?

Look at the `[tool_resp]` event: salary, ssn, home address — all `[REDACTED]` by the time they reach the model. The tool itself returned the full record — your Python still has it, for audit logs or systems that legitimately need it — but the model's context holds only what the question required.

The same shape covers truncating huge results, renaming fields the model finds confusing, or adding audit tags.

### 🎯 Mini-task

Add `"email"` to `SENSITIVE_FIELDS` and re-run. What does the agent say when it can't see the one field the question asked about? (This is a real design decision: redaction versus usefulness.)

# Mock a Tool for Tests

Last worry: how do you *test* an agent whose tool hits a paid API? You don't want every test run to cost money — but you want to test the real agent code, unmodified.

A **`before_tool_callback`** runs just before a tool executes. Return `None` and the tool runs; return a dict and the tool is skipped — the model receives your dict as if the tool had returned it.

```python
def mock_in_tests(tool, args, tool_context):
```

Three arguments, filled in by ADK: which tool is about to run, with what arguments, and the context. Our mock checks the tool name and the ticker: known ticker → return canned data (the tool never runs); unknown ticker → `None` (the real call goes ahead).

In [7]:
# A real API tool — expensive, external, we don't want it firing in tests.
def fetch_stock_price(ticker: str) -> dict:
    """Fetch the current stock price for a ticker."""
    # In production this would hit a real finance API.
    # Here we just return a plausible-looking fake to prove the callback is short-circuiting.
    print(f"   ↪ REAL fetch_stock_price called for {ticker}")
    return {"ticker": ticker, "price": 123.45, "currency": "USD", "source": "live"}

MOCK_RESPONSES = {
    "AAPL": {"ticker": "AAPL", "price": 180.00, "currency": "USD", "source": "mock"},
    "GOOG": {"ticker": "GOOG", "price": 140.00, "currency": "USD", "source": "mock"},
}

def mock_in_tests(tool, args, tool_context):
    """Return a mock response instead of hitting the real API."""
    if tool.name == "fetch_stock_price":
        ticker = args.get("ticker", "").upper()
        if ticker in MOCK_RESPONSES:
            print(f"   ↪ Short-circuiting fetch_stock_price for {ticker} with mock.")
            return MOCK_RESPONSES[ticker]
    return None  # Let the real tool run.

stock_agent = LlmAgent(
    name="stock_agent",
    model=LiteLlm(model=MODEL_STRING),
    description="Reports stock prices.",
    instruction=(
        "For every stock-related question, call fetch_stock_price with the ticker "
        "the user mentioned. Never answer without calling it. Report the numeric "
        "price and currency."
    ),
    tools=[fetch_stock_price],
    before_tool_callback=mock_in_tests,
)

await chat(stock_agent, "What's AAPL trading at?")
await chat(stock_agent, "What about MSFT?")

USER: What's AAPL trading at?


[tool_call] fetch_stock_price({'ticker': 'AAPL'})
   ↪ Short-circuiting fetch_stock_price for AAPL with mock.
[tool_resp] {'ticker': 'AAPL', 'price': 180.0, 'currency': 'USD', 'source': 'mock'}


[FINAL] stock_agent: AAPL is trading at **$180.00 USD**.

USER: What about MSFT?


[tool_call] fetch_stock_price({'ticker': 'MSFT'})
   ↪ REAL fetch_stock_price called for MSFT
[tool_resp] {'ticker': 'MSFT', 'price': 123.45, 'currency': 'USD', 'source': 'live'}


[FINAL] stock_agent: MSFT is **$123.45 USD**.



### 🔍 What just happened?

- AAPL is in `MOCK_RESPONSES` → the callback returned the mock; the `↪ REAL` print never fired.
- MSFT is not → the callback returned `None`; the real tool ran (you can see `↪ REAL` in the output).

This is how agent code becomes testable: mocks injected by a callback in tests, the callback left off in production — the same agent code on both paths.

### 🎯 Mini-tasks

1. **An audit log.** Write a `before_tool_callback` that appends every tool call to `tool_context.state["temp:tool_log"]`, then inspect the list after a run.
2. **A friendly error.** Make `fetch_stock_price` raise an exception, then add an `on_tool_error_callback` that returns `{"error": "Stock service unavailable"}`. Verify the agent answers gracefully instead of crashing.

# All Six Hooks at a Glance

You've now used the three most common hooks. Here are all six, with what each is good for:

| Hook | Fires | Common use |
|---|---|---|
| `before_agent_callback(ctx)` | Before the agent processes input | Pre-flight state setup; reject inputs on session-level conditions |
| `after_agent_callback(ctx)` | After the agent finishes | Final-output logging; write summary state |
| `before_model_callback(ctx, req)` | Before every LLM call | **Guardrails**, caching, prompt-injection checks |
| `after_model_callback(ctx, resp)` | After every LLM call | Response filtering; telemetry; cost tracking |
| `before_tool_callback(tool, args, ctx)` | Before every tool executes | **Mocking**, cache lookups, argument validation |
| `after_tool_callback(tool, args, ctx, resp)` | After every tool returns | **PII redaction**, result caching, error classification |

And the two error hooks: `on_model_error_callback` and `on_tool_error_callback` for custom error recovery (by default, the error propagates). All six follow return-to-override, and all six are plain Python functions passed in at agent construction.

# Which Mechanism When?

Callbacks are not the only way to step into an agent's behavior. Four mechanisms, four scopes:

| Mechanism | Scope | When |
|---|---|---|
| **Instruction prompt** | One agent | Soft preferences, style, default behavior |
| **Tool function code** | One tool | Guards on irreversible operations (the `delete_ticket` demo) |
| **Callback** | One agent's steps | Rules that apply to every step of one agent — guardrails, PII, caching |
| **Plugin** | Whole runner, all agents | Org-wide policies; audit logging |

Rule of thumb: **callbacks for agent-specific logic, plugins for app-wide policy.** A blocklist for one specialist agent → callback. A company-wide audit trail that must fire on every agent → plugin (`google.adk.plugins` — beyond this module).

# One Honest Gap: Callbacks Don't Show in Traces

If you inspect runs with a tracing tool (Cloud Trace, Langfuse, Arize), you will see the model calls, the tool calls, and the state changes — but **not** your callbacks. "blocklist_guardrail ran and returned None" produces no trace entry (verified through ADK 2.7; documented in Google's own ADK posts).

So if a callback makes policy decisions you need to see, instrument it yourself — a `print`, a log line, or explicit telemetry inside the function.

# Key Takeaways

- **Six doors:** before/after × agent/model/tool, plus two error hooks — all plain Python functions passed at agent construction, **without parentheses**.
- **Return-to-override:** `None` = proceed; a returned object = use this instead.
- `before_model_callback` = the guardrail door — on a block, the LLM is never called.
- `after_tool_callback` = the output filter — the model sees only what it needs.
- `before_tool_callback` = the mock/cache door — test without hitting real APIs.
- Callbacks for agent-specific logic; plugins for app-wide policy.
- Callbacks don't appear in traces — instrument them yourself.

# Next up — M08: Memory

Session state covers what an agent remembers *within* a conversation. M08 is about remembering *across* conversations, for weeks: a real database under sessions (SQLite), and `MemoryService` + `load_memory` for long-term recall.